# 06 - ΔQ(V) and the first baselines

Notebook 05 showed that capacity itself says little about lifetime at cycle 100. The paper's idea is to look at how the **whole discharge curve** changes instead:

ΔQ₁₀₀₋₁₀(V) = Q₁₀₀(V) − Q₁₀(V)

For every voltage on the fixed grid (2.0 to 3.5 V), take the discharge capacity at cycle 100 minus the one at cycle 10. Cells that will die early already show a bigger, more uneven change in this curve, even when their total capacity still looks fine.

The paper's simplest model uses one number per cell, log10(variance of ΔQ), and a straight line to predict log10(cycle life).

Only training cells are used to look at the feature and fit models. The primary test set is used once, for the final numbers. The later batch (secondary test) stays untouched until Stage 3.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.features import delta_q_all, log_var

cells = pd.read_csv("../data/cells_clean.csv").set_index("cell_id")
train = cells[cells["split"] == "train"]
test = cells[cells["split"] == "test_primary"]
print(len(train), "train cells,", len(test), "primary test cells")

41 train cells, 43 primary test cells


## ΔQ curves for all 124 cells

Cycle numbers here are real cycle numbers (the curve for cycle n is stored at position n - 1, see notebook 01). Computing it for every cell now, and saving it, so later notebooks don't need the 8 GB files.

In [2]:
curves, voltage = delta_q_all(set(cells.index), root="../")
curves = curves.loc[cells.index]
print(curves.shape, "| voltage grid", voltage.max(), "to", voltage.min(), "V")
print("missing values:", int(curves.isna().sum().sum()))

curves.columns = [f"v{k}" for k in range(curves.shape[1])]
curves.to_parquet("../data/delta_q_100_10.parquet")
np.save("../data/voltage_grid.npy", voltage)

(124, 1000) | voltage grid 3.5 to 2.0 V
missing values: 0
